# Astrophysical Models

This notebook demonstrates the astrophysical component of `toolscosmo`: empirical models
that connect dark matter halo properties to observable galaxy properties at high redshift.

**Contents**

1. **Star Formation Efficiency (SFE)** — $f_*(M_h, z)$, the fraction of accreted baryons
   converted into stars as a function of halo mass and redshift.
2. **Mass Accretion Rates** — halo mass accretion $\dot{M}(M_h, z)$ under different
   prescriptions (EXP and Hubble-scale).
3. **UV Magnitude – Halo Mass Relation** — how halo mass maps to UV absolute magnitude
   $M_\mathrm{AB}$.
4. **UV Luminosity Functions (UVLFs)** — number density of galaxies per UV magnitude bin,
   compared to HST observations.

The SFE is parameterised as a double power-law in halo mass (Park et al. 2019):
$$f_*(M_h) = \frac{2 f_0}{\left(\frac{M_h}{M_p}\right)^{-\gamma_1} + \left(\frac{M_h}{M_p}\right)^{-\gamma_2}}
\cdot \left[1 + \left(\frac{M_t}{M_h}\right)^{\gamma_3}\right]^{\gamma_4/\gamma_3}$$
with peak efficiency $f_0$ at pivot mass $M_p$, faint-end slope $\gamma_1$,
bright-end slope $\gamma_2$, and low-mass suppression below threshold $M_t$.

Two **mass accretion prescriptions** are compared:
- **EXP** — exponential growth $M(z) \propto e^{-\alpha z}$ (Fakhouri et al. 2010).
- **Hubble-scale** — accretion rate tied to the Hubble time (21cmFAST convention).

---
## 1. Setup and Helper Functions

`set_param(**kwargs)` builds a fresh `toolscosmo.par()` object with all astrophysical and
cosmological parameters.  `model_uvlfs(**kwargs)` calls `toolscosmo.UVLF` and returns the
full luminosity-function output dictionary.

We run two models to compare the two accretion prescriptions:
- `lf_EXP` — exponential accretion (`MA='EXP'`)
- `lf_21cmfast` — Hubble-scale accretion (`MA='21cmfast'`)

All other SFE parameters are held fixed so that the only difference between the two is the
mass accretion history.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import toolscosmo

plt.rcParams.update({'font.size': 12, 'lines.linewidth': 2})


def set_param(**kwargs):
    """Build a toolscosmo parameter object with astrophysical and cosmological settings."""
    par = toolscosmo.par(DE='cpl')
    par.file.ps = 'CAMB'

    Odm = kwargs.get('Odm', 0.266)
    Ob  = kwargs.get('Ob', 0.049)
    par.cosmo.Ob = Ob
    par.cosmo.Om = kwargs.get('Om', Odm + Ob)
    par.cosmo.s8 = kwargs.get('s8', 0.830)
    par.cosmo.ns = kwargs.get('ns', 0.963)
    par.cosmo.h0 = kwargs.get('h0', 0.673)
    par.DE.w0    = kwargs.get('w0', -1)
    par.DE.wa    = kwargs.get('wa', 0)

    MA = kwargs.get('MA', 'EXP')
    if MA.upper() == 'EXP':
        par.code.MA = 'EXP'
        par.MA.alpha_EXP = kwargs.get('MA_param', 0.79)
    elif MA.upper() == 'EPS':
        par.code.MA = 'EPS'
        par.MA.Q_EPS = kwargs.get('MA_param', 0.6)
    else:
        par.code.MA  = 'HUBBLEscale'
        par.MA.t_star = kwargs.get('MA_param', 0.5)

    par.code.NM      = 80
    par.code.Nz      = 100
    par.code.kmin    = 1e-5
    par.code.kmax    = 5e2
    par.code.Mmin    = 1e5
    par.code.Mmax    = 3e15
    par.code.zmin    = 5.0
    par.code.zmax    = 50.0
    par.code.verbose = kwargs.get('verbose', True)

    par.lf.Muv_min   = kwargs.get('Muv_min', -23)
    par.lf.Muv_max   = kwargs.get('Muv_max', -5)
    par.lf.NMuv      = kwargs.get('NMuv', 33)
    par.lf.sig_M     = kwargs.get('sig_M', 0.56)
    par.lf.eps_sys   = kwargs.get('eps_sys', 1)
    par.lf.f0_sfe    = kwargs.get('f0_sfe', 0.1)
    par.lf.Mp_sfe    = kwargs.get('Mp_sfe', 10**11.28)
    par.lf.g1_sfe    = kwargs.get('g1_sfe', 0.49)
    par.lf.g2_sfe    = kwargs.get('g2_sfe', -0.60)
    par.lf.Mt_sfe    = kwargs.get('Mt_sfe', 10**4)
    par.lf.g3_sfe    = kwargs.get('g3_sfe', 5.0)
    par.lf.g4_sfe    = kwargs.get('g4_sfe', -5.0)
    par.lf.f0_sfe_nu = kwargs.get('f0_sfe_nu', -0.58)
    par.lf.Mp_sfe_nu = 0.0
    par.lf.Mt_sfe_nu = 0.0
    par.lf.g1_sfe_nu = 0.0
    par.lf.g2_sfe_nu = 0.0
    par.lf.g3_sfe_nu = 0.0
    par.lf.g4_sfe_nu = 0.0
    return par


def model_uvlfs(**kwargs):
    """Compute the UV luminosity function and return the output dictionary."""
    par = kwargs.get('param', set_param(**kwargs))
    uvlf    = toolscosmo.UVLF(par)
    out_lf  = uvlf.UV_luminosity(f_duty=kwargs.get('f_duty', 1))
    return out_lf


# Run both accretion models with identical SFE parameters
sfe_kwargs = dict(g1_sfe=-0.5, g2_sfe=-0.5, g3_sfe=0, g4_sfe=0,
                  Mt_sfe=3.365e8*2, Mp_sfe=6.73e9, f_duty='EXP')

lf_EXP      = model_uvlfs(MA='EXP',      **sfe_kwargs)
lf_21cmfast = model_uvlfs(MA='21cmfast', MA_param=0.3, **sfe_kwargs)


---
## 2. Star Formation Efficiency $f_*(M_h, z)$

The SFE sets the stellar mass formed per unit accreted mass. It peaks at the pivot mass
$M_p \sim 10^{11}\,M_\odot$ and is suppressed at both low mass (photo-ionisation feedback)
and high mass (AGN feedback). The two line styles show the two accretion models —
differences in $f_*$ arise because the same star formation prescription is applied to
different accretion histories.

In [ ]:
zplot = [5, 8, 10]

fig, ax = plt.subplots(figsize=(6, 5))
for jj, zj in enumerate(zplot):
    z_jdx = np.abs(lf_EXP['z'] - zj).argmin()
    ax.loglog(lf_EXP['m'],      lf_EXP['fstar'][z_jdx, :],
              lw=5, c=f'C{jj}', alpha=0.3, ls='-',  label=f'z={zj} (EXP)')
    ax.loglog(lf_21cmfast['m'], lf_21cmfast['fstar'][z_jdx, :],
              lw=2, c=f'C{jj}', alpha=1.0, ls=':', label=f'z={zj} (Hubble-scale)')
ax.axis([1e5, 3e15, 5e-5, 8e-1])
ax.set_xlabel(r'$M_h$ [$h^{-1}M_\odot$]')
ax.set_ylabel(r'$f_*$')
ax.set_title('Star Formation Efficiency')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()


---
## 3. Mass Accretion Rates

The left panel shows the total accreted mass $M_\mathrm{accr}(z)$ for halos of different
present-day masses. The right panel shows the instantaneous accretion rate
$\dot{M}_\mathrm{accr}(z)$. The EXP prescription (solid, faded) gives faster early growth
than the Hubble-scale prescription (dotted), leading to differences in star formation
history at high redshift.

In [ ]:
Mplot = [1e7, 1e9, 1e11, 1e13]

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for jj, mj in enumerate(Mplot):
    m_jdx = np.abs(lf_EXP['m'] - mj).argmin()
    axs[0].semilogy(lf_EXP['z'],      lf_EXP['M_accr'][:, m_jdx],
                    lw=5, alpha=0.3, c=f'C{jj}', ls='-',
                    label='EXP' if jj == 0 else None)
    axs[0].semilogy(lf_21cmfast['z'], lf_21cmfast['M_accr'][:, m_jdx],
                    lw=2, alpha=1.0, c=f'C{jj}', ls=':',
                    label='Hubble-scale' if jj == 0 else None)
    axs[1].semilogy(lf_EXP['z'],      lf_EXP['dMdt_accr'][:, m_jdx],
                    lw=5, alpha=0.3, c=f'C{jj}', ls='-')
    axs[1].semilogy(lf_21cmfast['z'], lf_21cmfast['dMdt_accr'][:, m_jdx],
                    lw=2, alpha=1.0, c=f'C{jj}', ls=':')
axs[0].legend()
axs[0].set_xlabel('$z$'); axs[0].set_ylabel(r'$M_\mathrm{accr}$ [$h^{-1}M_\odot$]')
axs[0].set_title('Accreted mass')
axs[1].set_xlabel('$z$'); axs[1].set_ylabel(r'$\dot{M}_\mathrm{accr}$ [$h^{-1}M_\odot$ yr$^{-1}$]')
axs[1].set_title('Accretion rate')
plt.tight_layout(); plt.show()


---
## 4. UV Magnitude – Halo Mass Relation

The UV absolute magnitude $M_\mathrm{AB}$ is computed from the star formation rate via the
UV luminosity conversion factor $\kappa_\mathrm{UV}$. The left panel shows $M_\mathrm{AB}(M_h)$
at several redshifts; the right panel shows the ratio between the two accretion prescriptions,
which propagates the accretion-rate difference into the observable UV magnitude.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for jj, zj in enumerate(zplot):
    z_jdx = np.abs(lf_EXP['z'] - zj).argmin()
    axs[0].semilogx(lf_EXP['m'],      lf_EXP['M_AB'][z_jdx, :],
                    lw=5, c=f'C{jj}', alpha=0.3, ls='-',  label=f'z={zj} (EXP)')
    axs[0].semilogx(lf_21cmfast['m'], lf_21cmfast['M_AB'][z_jdx, :],
                    lw=2, c=f'C{jj}', alpha=1.0, ls=':', label=f'z={zj} (Hubble-scale)')
    axs[1].semilogx(lf_EXP['m'],
                    lf_21cmfast['M_AB'][z_jdx, :] / lf_EXP['M_AB'][z_jdx, :],
                    lw=2, c=f'C{jj}', ls=':', label=f'z={zj}')
axs[0].legend(fontsize=9)
axs[0].axis([1e5, 3e15, -26, 9])
axs[0].set_xlabel(r'$M_h$ [$h^{-1}M_\odot$]'); axs[0].set_ylabel(r'$M_\mathrm{AB}$')
axs[0].set_title('UV magnitude vs halo mass')
axs[1].axhline(1, color='k', lw=0.8, ls='--')
axs[1].axis([1e5, 3e15, -6, 9])
axs[1].set_xlabel(r'$M_h$ [$h^{-1}M_\odot$]'); axs[1].set_ylabel('Ratio (Hubble-scale / EXP)')
axs[1].set_title(r'$M_\mathrm{AB}$ ratio')
axs[1].legend(fontsize=9)
plt.tight_layout(); plt.show()


---
## 5. UV Luminosity Functions compared to observations

The UVLF $\phi(M_\mathrm{UV})$ is computed by convolving the halo mass function with the
$M_h \to M_\mathrm{UV}$ relation, including a log-normal scatter $\sigma_M$ in UV magnitude
at fixed halo mass.

The observational data shown (Park et al. 2019 compilation) cover $z \sim 6$–10 from
Bouwens+2015/2017, Atek+2018, Livermore+2017, Ishigaki+2017, and Oesch+2017.

> **Data files**: the comparison below loads pre-bundled text files from the `data/` folder.
> If not present, the model curve is shown without data points.

In [2]:
lstyles = ['-', '--', '-.', ':']

# Parameter setup — use CAMB instead of a pre-computed file
par = toolscosmo.par()
par.file.ps = 'CAMB'

par.code.zmin = 5
par.code.zmax = 40
par.code.Nz   = 50
par.code.Mmin = 5e5
par.code.Mmax = 2e15
par.code.NM   = 100
par.code.MA   = 'EXP'

par.mf.window = 'tophat'
par.mf.c      = 2.5
par.mf.q      = 0.85
par.mf.p      = 0.3

par.lf.f0_sfe    = 0.1
par.lf.g1_sfe    = 0.49
par.lf.g2_sfe    = -0.61
par.lf.g3_sfe    = 5
par.lf.g4_sfe    = -5
par.lf.Mp_sfe    = 2.0e11
par.lf.Mt_sfe    = 7e8
par.lf.f0_sfe_nu = -0.1
par.lf.g1_sfe_nu = par.lf.g2_sfe_nu = par.lf.g3_sfe_nu = par.lf.g4_sfe_nu = 0
par.lf.Mp_sfe_nu = par.lf.Mt_sfe_nu = 0
par.lf.Muv_min   = -23.
par.lf.Muv_max   = -8.0
par.lf.NMuv      = 20
par.lf.sig_M     = 0.2
par.lf.eps_sys   = 1.0

uvlf   = toolscosmo.UVLF(par)
out_lf = uvlf.UV_luminosity()

# Observational data (Park et al. 2019 compilation)
import os
data_dir = 'data'
obs_sets = [
    ('Park19_Bouwens.txt',   'o', 'coral',     'Bouwens+2015/17'),
    ('Park19_Atek.txt',      'D', 'orange',    'Atek+2018'),
    ('Park19_Livermore.txt', 's', 'lightblue', 'Livermore+2017'),
    ('Park19_Ishigaki.txt',  'p', 'violet',    'Ishigaki+2017'),
    ('Park19_Oesch.txt',     'h', 'green',     'Oesch+2017'),
]

fig, axs = plt.subplots(1, 4, figsize=(15, 4))
for j, zj in enumerate([6, 7, 8, 10]):
    axs[j].set_title(f'$z\\sim {zj}$')
    for fname, marker, color, label in obs_sets:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            continue
        data = np.loadtxt(fpath)
        az = np.where(data[:, -1] == zj)
        xx, yy = data[az, 0].squeeze(), data[az, 1].squeeze()
        yl, yp = data[az, 2].squeeze(), data[az, 3].squeeze()
        axs[j].errorbar(xx, yy, yerr=[yy - yl, yp - yy],
                        ls=' ', marker=marker, markeredgecolor='k', color=color,
                        label=label if j == 0 else None)
    z_jdx = np.abs(out_lf['z'] - zj).argmin()
    axs[j].plot(out_lf['uvlf']['Muv_mean'], out_lf['uvlf']['phi_uv'][z_jdx, :],
                lw=2.5, ls='-', color='k', label='Model' if j == 0 else None)
    axs[j].set_yscale('log')
    axs[j].axis([-23, -9, 7e-6, 3e2])
    axs[j].set_xlabel(r'$M_{\rm UV}$')
    axs[j].set_ylabel(r'$\phi\,(M_{\rm UV})$ [Mpc$^{-3}$ mag$^{-1}$]')
    if j == 0:
        axs[j].legend(fontsize=8)
plt.tight_layout()
plt.show()
